# 06 · seaborn：一行画一张统计图

> 目标不是背 API，是知道**什么时候用 seaborn 比 matplotlib 省事**。

这一节要带走三样：

1. `sns.histplot(data=df, x="列名")` —— 画直方图，不用自己传 Series
2. `hue="列名"` —— 一行代码按类别分组上色
3. `sns.boxplot` / `sns.relplot` —— 看分布、看关系，各一行

其余参数一律不用记，用的时候查文档。

## 0. 准备

数据还是项目一那份 Titanic。注意路径是 `../data/raw/`，因为 notebook 在 `notebooks/` 里。

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False  # 不设的话负号会变成方框
plt.rcParams["figure.dpi"] = 110

df = pd.read_csv("../data/raw/titanic.csv")

print("行数:", len(df))
print("列名:", list(df.columns))
df.head(3)

## 1. 同一张图，两种写法

matplotlib 是"画笔"，你要自己给数据、自己写标题；
seaborn 建在它上面，**认识 DataFrame，认识列名**。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["Fare"], bins=30)
axes[0].set_title("matplotlib")

sns.histplot(data=df, x="Fare", bins=30, ax=axes[1])
axes[1].set_title("seaborn")

plt.tight_layout()

**观察点**：两张图的形状一模一样，但 seaborn 那半边**自动给坐标轴加了标签**
（x 轴写 Fare、y 轴写 Count）。这就是"高层封装"——它知道你在画什么，所以能替你猜。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, mode, title in zip(
    axes,
    ["layer", "dodge", "stack"],
    [
        "multiple='layer'（默认，叠加）",
        "multiple='dodge'（并排）",
        "multiple='stack'（堆叠）",
    ],
):
    sns.histplot(data=df, x="Fare", bins=30, hue="Sex", multiple=mode, ax=ax)
    ax.set_title(title)
    ax.set_xlim(0, 200)  # 票价长尾，截一下才看得清

plt.tight_layout()

这才是 seaborn 真正省事的地方。同样的"按性别分组的票价分布"：

观察点：seaborn 那半边代码短，而且自动生成了图例。

但要注意：两组柱子并不是并排的，而是叠在一起画（重叠处颜色变深）。这是 histplot 的默认行为，参数叫 multiple，默认值是 layer。

想真正画成并排，要显式写 multiple="dodge"：

sns.histplot(data=df, x="Fare", bins=30, hue="Sex", multiple="dodge")
三种模式各有用途：

layer（默认）：看两组各自的分布，但重叠处会互相干扰
dodge：两组左右分列，比高度最直观
stack：柱子高度是两组之和，用来看总量和构成
下一格把三种画在一起对照。

In [ ]:
# 你的代码：并排画 Age 的直方图
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["Age"], bins=30)
axes[0].set_title("matplotlib")

sns.histplot(data=df, x="Age", bins=30, ax=axes[1])
axes[1].set_title("seaborn")

plt.tight_layout()

## 2. `hue`：一行代码按类别分组

这才是 seaborn 真正省事的地方。同样的"按性别分组的票价分布"：

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# matplotlib：要自己分组、自己循环、自己加图例
for sex, group in df.groupby("Sex"):
    axes[0].hist(group["Fare"], bins=30, alpha=0.6, label=sex)
axes[0].legend()
axes[0].set_title("matplotlib：自己写循环")

# seaborn：加一个 hue 参数
sns.histplot(data=df, x="Fare", bins=30, hue="Sex", multiple="dodge", ax=axes[1])
axes[1].set_title("seaborn：hue 一行搞定")

plt.tight_layout()

**观察点**：seaborn 那半边不只是代码短——它把两组柱子画成**并排**的，
还自动加了图例。matplotlib 那边是半透明叠加，读者很难直接比较。

`hue` 的意思是"用这一列给数据上色"，是 seaborn 里最常用的参数，值得记住。

### 🔹 小练习 2 · 五分钟

把上面 seaborn 那半边改成**按舱位（`Pclass`）分组**。

跑完看一眼：一等舱和三等舱的票价分布差多少？这个差别和项目一里的结论对得上吗？

In [ ]:
# 你的代码：按 Pclass 分组的票价直方图
fig, axes = plt.subplots(figsize=(12, 4))

sns.histplot(data=df, x="Fare", bins=30, hue="Pclass", multiple="dodge", ax=axes)
axes.set_title("seaborn：按 Pclass 分组的票价直方图")

## 3. 箱线图：看分布，不看数量

直方图告诉你"有多少人"，箱线图告诉你"分布长什么样"——
中位数在哪、中间 50% 的范围有多宽、有没有离群点。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(data=df, x="Pclass", y="Fare", ax=axes[0])
axes[0].set_title("箱线图")

sns.violinplot(data=df, x="Pclass", y="Fare", ax=axes[1])
axes[1].set_title("小提琴图")

plt.tight_layout()

**怎么读箱线图**（从左到右）：左边的须 = 最小值（不含离群点），箱子的左边 = 25% 分位，
箱子中间的线 = **中位数**，箱子右边 = 75% 分位，右边的须 = 最大值，散落的点 = 离群值。

小提琴图是箱线图的升级版：宽度表示那个价位的人有多少。信息量更大，但更难读。

### 🔹 小练习 3 · 五分钟

用箱线图看 **Age × Survived**：活下来的人和没活下来的人，年龄分布有差别吗？

提示：`x="Survived"`，`y="Age"`。看完自己判断一句："这个差别算大还是算小？"

In [ ]:
# 你的代码：Age 按 Survived 分组的箱线图
fig, axes = plt.subplots(figsize=(12, 4))

sns.boxplot(data=df, x="Survived", y="Age", ax=axes)
axes.set_title("seaborn：Age 按 Survived 分组的箱线图")
plt.tight_layout()

## 4. 一张图看四个变量：`relplot`

`relplot` 是"关系图"的工厂：`hue` 上色、`col` 分面（切成左右两张）、`size` 改点大小。

In [ ]:
sns.relplot(
    data=df,
    x="Age",
    y="Fare",
    hue="Survived",
    col="Sex",
    height=4,
)

**观察点**：一张图里同时表达了年龄、票价、是否生还、性别四个变量。
代价是——Fare 有几个极大的离群值，把大部分点压在了底部。

这正好是项目一里那个教训的再现：**右偏的数据直接画散点，主角会被离群值挤走**。

### 🔹 小练习 4 · 五分钟

把上面那张图改成**只画票价小于 200 的乘客**，看看画面有什么变化。

提示：先 `df[df["Fare"] < 200]` 筛一份数据，再传给 `relplot`。

想一想：这样做丢掉了多少条数据？丢掉它们合理吗？

In [ ]:
# 你的代码
sns.relplot(
    data=df[df["Fare"] < 200], x="Age", y="Fare", hue="Survived", col="Sex", height=4
)
plt.tight_layout()

print("原始数据行数:", len(df))
print("筛选后数据行数:", len(df[df["Fare"] < 200]))
print("丢掉了", len(df) - len(df[df["Fare"] < 200]), "行数据")
print("占比:", (len(df) - len(df[df["Fare"] < 200])) / len(df) * 100, "%")

# 丢了20行数据，占总数据的2.5%，丢掉他们合理，丢了之后图就没被极端值压缩在一起了，

## 5. 大练习：用 seaborn 重画项目一第 5 题的图

项目一第 5 题你画过一张"能讲故事的图"。现在用 seaborn 再画一次，要求：

1. 用 `hue` 表达一个分类变量（比如 Survived 或 Pclass）；
2. 图里能一眼看出"谁更容易活下来"；
3. 标题写成一句**结论**，不是"某某图"。

做完对比一下：seaborn 版本比 matplotlib 版本少写了多少行？

In [ ]:
# 你的代码：一张能讲故事的图（seaborn 版）
fig, axes = plt.subplots(figsize=(8, 6))

sns.barplot(data=df, x="Pclass", y="Survived", hue="Sex", ax=axes)
axes.set_title("三等舱女性的生还率（50%）仍高于头等舱男性（37%）")

## 6. 总结卡片

| 想要什么 | 一行代码 |
| :--- | :--- |
| 直方图 | `sns.histplot(data=df, x="列")` |
| 按类别分组上色 | 加 `hue="列"` |
| 箱线图 | `sns.boxplot(data=df, x="类别", y="数值")` |
| 小提琴图 | `sns.violinplot(...)`，参数同上 |
| 散点 + 分组 + 分面 | `sns.relplot(data=df, x=..., y=..., hue=..., col=...)` |
| 一次性看所有数值列 | `sns.pairplot(df)` （慢，别对大数据用） |

**什么时候用 seaborn、什么时候用 matplotlib**：

- 统计图（分布、分组对比、关系）→ seaborn，省事
- 精细控制（自定义标注、箭头、多个坐标轴）→ matplotlib
- 两者可以混用：seaborn 画图，matplotlib 调细节

## 7. 卡住时的提示

**报错 `KeyError`**：列名写错了。先 `print(df.columns)` 看一眼真实列名，注意大小写。

**图不显示**：notebook 里最后一行写 `plt.tight_layout()` 或直接留空都行，
但如果你在外面写了别的语句，可能要补一个 `plt.show()`。

**图挤在一起**：加 `plt.tight_layout()`。

**两张图叠在一起了**：每张图前面加一句 `plt.figure()` 或重新 `plt.subplots()`。